# Giustificazioni verificabili e ritiro su LUBM

Benchmark con **numero di query configurabile**. Impostare `N_QUERIES = "all"` per tutte
le query ammissibili oppure un intero positivo, per esempio `25`.
Le query positive provengono da `owlapy_generated_qa.json`; HermiT verifica l'entailment
originale e il non-entailment dopo il ritiro di un singolo assioma.

Il builder verifica tutto il pool prima di selezionare N casi senza ripetizioni, con seed fisso.
Non impone quote per profondità o soggetti distinti. Se N supera i casi disponibili, dà un errore
esplicito senza produrre un dataset ridotto. Ogni nuovo dataset è congelato in una directory propria;
il dataset `subclass_grounding_v1` da sei casi rimane invariato.

Ogni query comporta due chiamate: **intera ontologia originale**, poi **intera ontologia aggiornata**,
con la risposta autentica precedente nella cronologia. Le coppie sono indipendenti.
Il costo è `2 × N × numero di modelli` chiamate. La selezione e i controlli HermiT non chiamano Ollama.

## Ambiente e avvio

Java nel PATH, Ollama avviato e un modello già installato. Kernel **Python (epistemic pilot)**.

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-belief-revision-full.txt
```

Configurare la prossima cella. Eseguire solo le celle prima del ciclo su `MODELS` per preparare
e verificare il dataset; **Run All** esegue anche il benchmark LLM. Nessun modello viene scaricato.
Gli output del precedente pilot restano nel run storico
`generated/subclass_grounding/20260929T125224Z_cbd1a24f/`; sono stati rimossi dalle celle aggiornate
per non confonderli con il nuovo dataset.


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
ROOT = next((p.resolve() for p in roots if (p / "subclass_grounding_benchmark.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from subclass_grounding_benchmark import OPTIONS, build_dataset, load_dataset, run_benchmark, compare_runs, message

MODELS = ["qwen2.5-coder:7b"]  # aggiungere solo nomi esatti di modelli locali
N_QUERIES = "all"  # oppure, per esempio, 25
SELECTION_SEED = 20260929
suffix = "" if SELECTION_SEED == 20260929 else f"_seed{SELECTION_SEED}"
DATASET_DIR = ROOT / "benchmarks" / f"subclass_grounding_v2_{N_QUERIES}{suffix}"
if not DATASET_DIR.exists():
    build_dataset(DATASET_DIR, seed=SELECTION_SEED, n_queries=N_QUERIES)
data, graphs, texts, oracle, dataset_hash = load_dataset(DATASET_DIR)
if data.get("requested_n_queries") != N_QUERIES or data["seed"] != SELECTION_SEED:
    raise ValueError("Il dataset esistente ha una configurazione diversa. Usare una directory nuova.")
print("Casi ammissibili:", data["eligible_queries"], "| selezionati:", len(data["cases"]))
print("Distribuzione per profondità:", data["selected_by_depth"])
print("Chiamate previste:", 2 * len(data["cases"]) * len(MODELS))
print("Dataset:", data["version"], "| SHA-256:", dataset_hash)
print("Casi:", len(data["cases"]), "| opzioni fisse:", OPTIONS)
table = "| Caso | Query | Passaggi nella prova esplicita | Assioma ritirato |\n|---|---|---|---|"
for case in data["cases"]:
    short = lambda uri: uri.split("#")[-1]
    removed = case["removed_axioms"][0]
    table += f"\n| {case['id']} | {short(case['query'][0])} ⊑ {short(case['query'][1])} | {case['explicit_proof_depth']} | {short(removed['subject'])} ⊑ {short(removed['object'])} |"
display(Markdown(table))

Casi ammissibili: 52 | selezionati: 52
Distribuzione per profondità: {'1': 33, '2': 15, '3': 4}
Chiamate previste: 104
Dataset: subclass_grounding_v2_all | SHA-256: 8710b0ce845545514c6aba52a076062c48982e1802a863463457e034faf9db3b
Casi: 52 | opzioni fisse: {'temperature': 0, 'seed': 42, 'num_ctx': 32768, 'num_predict': 768}


| Caso | Query | Passaggi nella prova esplicita | Assioma ritirato |
|---|---|---|---|
| subclass_01 | AdministrativeStaff ⊑ Employee | 1 | AdministrativeStaff ⊑ Employee |
| subclass_02 | Article ⊑ Publication | 1 | Article ⊑ Publication |
| subclass_03 | AssistantProfessor ⊑ Professor | 1 | AssistantProfessor ⊑ Professor |
| subclass_04 | AssociateProfessor ⊑ Professor | 1 | AssociateProfessor ⊑ Professor |
| subclass_05 | Book ⊑ Publication | 1 | Book ⊑ Publication |
| subclass_06 | Chair ⊑ Professor | 1 | Chair ⊑ Professor |
| subclass_07 | ClericalStaff ⊑ AdministrativeStaff | 1 | ClericalStaff ⊑ AdministrativeStaff |
| subclass_08 | College ⊑ Organization | 1 | College ⊑ Organization |
| subclass_09 | ConferencePaper ⊑ Article | 1 | ConferencePaper ⊑ Article |
| subclass_10 | Course ⊑ Work | 1 | Course ⊑ Work |
| subclass_11 | Dean ⊑ Professor | 1 | Dean ⊑ Professor |
| subclass_12 | Department ⊑ Organization | 1 | Department ⊑ Organization |
| subclass_13 | Faculty ⊑ Employee | 1 | Faculty ⊑ Employee |
| subclass_14 | FullProfessor ⊑ Professor | 1 | FullProfessor ⊑ Professor |
| subclass_15 | GraduateCourse ⊑ Course | 1 | GraduateCourse ⊑ Course |
| subclass_16 | GraduateStudent ⊑ Person | 1 | GraduateStudent ⊑ Person |
| subclass_17 | Institute ⊑ Organization | 1 | Institute ⊑ Organization |
| subclass_18 | JournalArticle ⊑ Article | 1 | JournalArticle ⊑ Article |
| subclass_19 | Lecturer ⊑ Faculty | 1 | Lecturer ⊑ Faculty |
| subclass_20 | Manual ⊑ Publication | 1 | Manual ⊑ Publication |
| subclass_21 | PostDoc ⊑ Faculty | 1 | PostDoc ⊑ Faculty |
| subclass_22 | Professor ⊑ Faculty | 1 | Professor ⊑ Faculty |
| subclass_23 | Program ⊑ Organization | 1 | Program ⊑ Organization |
| subclass_24 | Research ⊑ Work | 1 | Research ⊑ Work |
| subclass_25 | ResearchGroup ⊑ Organization | 1 | ResearchGroup ⊑ Organization |
| subclass_26 | Software ⊑ Publication | 1 | Software ⊑ Publication |
| subclass_27 | Specification ⊑ Publication | 1 | Specification ⊑ Publication |
| subclass_28 | SystemsStaff ⊑ AdministrativeStaff | 1 | SystemsStaff ⊑ AdministrativeStaff |
| subclass_29 | TechnicalReport ⊑ Article | 1 | TechnicalReport ⊑ Article |
| subclass_30 | UndergraduateStudent ⊑ Student | 1 | UndergraduateStudent ⊑ Student |
| subclass_31 | University ⊑ Organization | 1 | University ⊑ Organization |
| subclass_32 | UnofficialPublication ⊑ Publication | 1 | UnofficialPublication ⊑ Publication |
| subclass_33 | VisitingProfessor ⊑ Professor | 1 | VisitingProfessor ⊑ Professor |
| subclass_34 | AssistantProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_35 | AssociateProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_36 | Chair ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_37 | ClericalStaff ⊑ Employee | 2 | AdministrativeStaff ⊑ Employee |
| subclass_38 | ConferencePaper ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_39 | Dean ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_40 | FullProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_41 | GraduateCourse ⊑ Work | 2 | Course ⊑ Work |
| subclass_42 | JournalArticle ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_43 | Lecturer ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_44 | PostDoc ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_45 | Professor ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_46 | SystemsStaff ⊑ Employee | 2 | AdministrativeStaff ⊑ Employee |
| subclass_47 | TechnicalReport ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_48 | VisitingProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_49 | AssistantProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_50 | AssociateProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_51 | FullProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_52 | VisitingProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |

## Output richiesto e verifica formale

Il modello restituisce soltanto `answer` (`entailed` / `not_entailed`) e `axioms`:

```json
{
  "answer": "entailed",
  "axioms": [
    {
      "subject": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Course",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "object": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Work"
    }
  ]
}
```

L'esempio sopra illustra il formato al lettore del notebook; non è inserito nei prompt.
Con `not_entailed`, `axioms` deve essere `[]`.

Per una risposta positiva con insieme di assiomi citati (J), snapshot (O) e query (q),
verifichiamo:

- correttezza della risposta rispetto a HermiT su (O);
- **provenienza:** (J \subseteq O), quindi ogni assioma è realmente asserito nello snapshot;
- **sufficienza:** (J \models q), tramite HermiT sul solo supporto citato;
- minimalità per inclusione, registrata separatamente e non richiesta per superare il caso.

Una prova alternativa valida passa: non facciamo exact match con la prova di riferimento.
Citare una conclusione inferita come se fosse un assioma asserito non passa.

Per una risposta negativa, la lista vuota è una convenzione di output e non una prova:
il non-entailment è verificato sullo snapshot completo. Nessun testo libero viene richiesto,
quindi questo esperimento misura il supporto strutturato dichiarato e non la coerenza di una spiegazione narrativa.

## Score

Per ogni coppia (i):

- (G_i=1) se la risposta originale è corretta e il suo supporto è presente e sufficiente; altrimenti 0.
- (R_i=1) se dopo il ritiro la risposta è `not_entailed` con lista vuota; altrimenti 0.
- (P_i=G_iR_i): successo sull'intera coppia.

Riportiamo **grounded score** (\sum G_i/N), **withdrawal score** (\sum R_i/N),
e **paired grounded revision score** (\sum P_i/N). Quest'ultimo è lo score congiunto principale.
Il denominatore è sempre costituito da tutte le N query selezionate: non escludiamo le baseline sbagliate.
Riportiamo anche l'accuracy iniziale, le risposte iniziali corrette prive di supporto valido
e l'eventuale riuso di assiomi ritirati.

Errori di formato del modello contano come insuccessi. Errori infrastrutturali, di contesto o
chiamate mancanti rendono il run **non confrontabile**, anziché abbassare arbitrariamente lo score.
I dettagli e gli insuccessi restano nei log. La finestra fissa è 32768 token; il budget conservativo
e la capacità riportata da Ollama vengono controllati senza tagliare il contesto.

In [4]:
# Ispezionare questa variabile per vedere l'intero prompt, senza chiamare il modello.
example_request = message(data["cases"][0], "original", texts)
print("Caratteri nel primo messaggio:", len(example_request["content"]))
print("Triple nello snapshot originale:", len(graphs["ontology_original.ttl"]))


Caratteri nel primo messaggio: 11789
Triple nello snapshot originale: 319


## esecuzione


### gemma4:e4b


In [7]:
run_dirs = []
results = {}
model = "gemma4:e4b"
print("\nMODELLO:", model, flush=True)
directory, result = run_benchmark(model, DATASET_DIR)
run_dirs.append(directory)
results[model] = result


MODELLO: gemma4:e4b
subclass_01 original {'answer': 'not_entailed', 'axioms': []}
subclass_01 withdrawal {'answer': 'not_entailed', 'axioms': []}
subclass_02 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Publication', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Article'}]}
subclass_02 withdrawal {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Publication', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Article'}]}
subclass_03 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor'}]}
subclass_03 withdrawal {'answer': 'not_entailed', 'ax

In [8]:
for model, result in results.items():
    display(Markdown(f"### {model}"))
    summary = result["summary"]
    display(Markdown(
        "| Misura | Valore |\n|---|---|\n" + "\n".join(
            f"| {key} | {summary[key]} |" for key in (
                "complete_for_comparison", "paired_grounded_revision_score", "original_grounded_score",
                "withdrawal_score", "original_answer_accuracy", "correct_original_without_valid_support_count"))))
    print("Score per profondità:", json.dumps(result["by_depth"], indent=2))
    table = "| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |\n|---|---|---|---|"
    for pair in result["pairs"]:
        table += f"\n| {pair['case_id']} | {pair['original_success']} | {pair['withdrawal_success']} | {pair['pair_success']} |"
    display(Markdown(table))
    for row in result["responses"]:
        print(row["case_id"], row["condition"], "risposta:", row["answer"],
              "atteso:", row["gold"], "supporto sufficiente:", row["support_sufficient"])
print("Cartelle dei risultati:")
for directory in run_dirs:
    print(directory)

### gemma4:e4b

| Misura | Valore |
|---|---|
| complete_for_comparison | True |
| paired_grounded_revision_score | 0.75 |
| original_grounded_score | 0.8653846153846154 |
| withdrawal_score | 0.8076923076923077 |
| original_answer_accuracy | 0.8653846153846154 |
| correct_original_without_valid_support_count | 0 |

Score per profondità: {
  "1": {
    "n_pairs": 33,
    "original_grounded_score": 0.9696969696969697,
    "withdrawal_score": 0.9696969696969697,
    "paired_grounded_revision_score": 0.9393939393939394
  },
  "2": {
    "n_pairs": 15,
    "original_grounded_score": 0.8,
    "withdrawal_score": 0.6,
    "paired_grounded_revision_score": 0.5333333333333333
  },
  "3": {
    "n_pairs": 4,
    "original_grounded_score": 0.25,
    "withdrawal_score": 0.25,
    "paired_grounded_revision_score": 0.0
  }
}


| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |
|---|---|---|---|
| subclass_01 | False | True | False |
| subclass_02 | True | False | False |
| subclass_03 | True | True | True |
| subclass_04 | True | True | True |
| subclass_05 | True | True | True |
| subclass_06 | True | True | True |
| subclass_07 | True | True | True |
| subclass_08 | True | True | True |
| subclass_09 | True | True | True |
| subclass_10 | True | True | True |
| subclass_11 | True | True | True |
| subclass_12 | True | True | True |
| subclass_13 | True | True | True |
| subclass_14 | True | True | True |
| subclass_15 | True | True | True |
| subclass_16 | True | True | True |
| subclass_17 | True | True | True |
| subclass_18 | True | True | True |
| subclass_19 | True | True | True |
| subclass_20 | True | True | True |
| subclass_21 | True | True | True |
| subclass_22 | True | True | True |
| subclass_23 | True | True | True |
| subclass_24 | True | True | True |
| subclass_25 | True | True | True |
| subclass_26 | True | True | True |
| subclass_27 | True | True | True |
| subclass_28 | True | True | True |
| subclass_29 | True | True | True |
| subclass_30 | True | True | True |
| subclass_31 | True | True | True |
| subclass_32 | True | True | True |
| subclass_33 | True | True | True |
| subclass_34 | True | True | True |
| subclass_35 | True | True | True |
| subclass_36 | True | True | True |
| subclass_37 | False | True | False |
| subclass_38 | True | False | False |
| subclass_39 | True | True | True |
| subclass_40 | True | True | True |
| subclass_41 | True | True | True |
| subclass_42 | True | False | False |
| subclass_43 | True | False | False |
| subclass_44 | True | True | True |
| subclass_45 | False | False | False |
| subclass_46 | False | False | False |
| subclass_47 | True | False | False |
| subclass_48 | True | True | True |
| subclass_49 | False | True | False |
| subclass_50 | False | False | False |
| subclass_51 | True | False | False |
| subclass_52 | False | False | False |

subclass_01 original risposta: not_entailed atteso: entailed supporto sufficiente: None
subclass_01 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_02 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_02 withdrawal risposta: entailed atteso: not_entailed supporto sufficiente: False
subclass_03 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_03 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_04 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_04 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_05 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_05 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_06 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_06 withdrawal ri

## Confrontare i modelli e leggere i limiti

I risultati includono anche gli score separati per profondità della prova.

`compare_runs` accetta solo run completi con lo stesso fingerprint di dataset, istruzioni, schema,
opzioni ed evaluator. Si possono aggiungere cartelle di run precedenti alla lista della prossima cella.
Il codice aggiornato ha un fingerprint diverso dal pilot storico: rieseguire i modelli con lo stesso codice per nuovi confronti.
Digest del modello, versione Ollama, Java, librerie e piattaforma sono registrati nel manifest.
Per confronti sperimentali controllati mantenere anche ambiente e hardware costanti, o documentare le differenze.

Il numero di query non garantisce da solo una graduatoria affidabile. Le query condividono
parte degli assiomi e alcune ritirano lo stesso collegamento: non sono prove indipendenti.
Le query che richiedono assiomi complessi (restrizioni, equivalenze) senza una prova di sole inclusioni
nominate sono fuori dallo scope di questa prima versione, pur restando quei costrutti nel contesto.

In questo protocollo tutti i secondi turni sono negativi. Un modello che impara semplicemente a
cambiare risposta può quindi ottenere un alto score: serviranno controlli senza variazione di entailment
per una misura più forte della revisione selettiva. Qui manteniamo deliberatamente il primo test semplice
richiesto. Non interpretiamo una prova valida come evidenza del processo interno realmente seguito.

In [9]:
# Aggiungere qui altre cartelle di run per un confronto sul medesimo protocollo.
comparison_dirs = list(run_dirs)
if all(result["summary"]["complete_for_comparison"] for result in results.values()):
    comparison = compare_runs(comparison_dirs)
    for row in comparison:
        print(row["model"], "| coppie:", row["pair_success_count"], "/", row["n_pairs"],
              "| score:", row["paired_grounded_revision_score"], "| digest:", row["digest"])
else:
    print("Almeno un run è incompleto: nessuna graduatoria generata.")

gemma4:e4b | coppie: 39 / 52 | score: 0.75 | digest: c6eb396dbd5992bbe3f5cdb947e8bbc0ee413d7c17e2beaae69f5d569cf982eb
